## Tutorial

this notebook is a simple implementation of the finetuning technique using UnSloth for faster inferencing.

---

FineTuning is a process to train a model, specially the LLMs for a specific need and task. This process is built over the pre existing knowledge of the model,  enhancing performance on specific tasks with reduced data and computational
requirements.

Fine-tuning transfers the pre-trained model’s learned patterns and features to new tasks, improving
performance and reducing training data needs. It has become popular in NLP for tasks like text classification, sentiment analysis, and question-answering.

### Stages of FineTuning

1. Stage 1: DataSet Prep

- this process involves the cleaning and preprocessing of the data used for the fineTuning of the model depending on the usecases.
- For example, in instruction tuning, the dataset may look like:
  - ###Human: $<Input Query>$
  - ###Assistant: $<Generated Output>$

2. Stage 2: Model Initialisation

- This process includes the basic setup for the finetuning, like the env., deps, and gpu and tpu selection (Colab) and many more basic requirement.

3. Stage 3: Training Setup
Till this stage we are mostly configured the tools and the functionalities we need to use.

- Define the env. similar to stage 2.
- Define and setting the HyperParams.
  - HyperParams include
    - a.) Learning Rate - The Optimisation and selection of the Loss function.
    - b.) Epochs - Epochs refers to the full pass of the model through the dataset once.
    - c.) Batch size - A batch refers to a subset of the training data used to update a model’s weights
during the training process. Batch training involves dividing the entire training set into smaller
groups, updating the model after processing each batch.

  - There are Variuos methods and optimisation methodologies for the hyperparams tuning that led to the healty finetning.



4. Stage 4: Selection of Fine-Tuning Techniques and Configuration

- Once the environment and hyperparameters are set, the next critical phase is choosing exactly how the model's weights will be updated during training. This stage defines the core mechanics of your fine-tuning pipeline.

  - **Full vs. Partial Fine-Tuning:**
    - *Full Fine-Tuning:* Involves updating every parameter in the base model. It yields strong task-specific performance but is highly resource-intensive and computationally expensive.
    - *Partial Fine-Tuning:* Updates only a specific subset of parameters, drastically reducing memory requirements and training time.


  - **Parameter-Efficient Fine-Tuning (PEFT):** This is a subset of partial fine-tuning and is the foundation of tools like Unsloth. Methods include:
    - *Adapters:* Inserting small, trainable layers within the existing neural network architecture.
    - *LoRA (Low-Rank Adaptation):* Instead of updating the massive dense matrices directly, LoRA injects smaller, trainable low-rank decomposition matrices into the model.
    - *QLoRA & DoRA:* Advanced variations of LoRA that incorporate quantization (QLoRA) or weight decomposition (DoRA) for even better efficiency on consumer hardware.


  - **Alignment Methodologies (Optional):** Depending on the use case, advanced optimization can be applied here to align the model's behavior.
    - *PPO (Proximal Policy Optimisation):* Uses reinforcement learning to train the model based on human feedback.
    - *DPO (Direct Preference Optimisation):* A more stable alternative to PPO for aligning the model with human preferences.



5. Stage 5: Evaluation and Validation

- After the training loop completes, the model must be rigorously tested before it can be relied upon.

  - **Validation Testing:** This involves running the newly trained model against a separate, holdout dataset. This step ensures the model has actually learned the specific task and isn't just memorizing (overfitting) the training data.
  - **Preventing Catastrophic Forgetting:** You must evaluate the model to ensure that while it learned the new specialized task, it did not overwrite or "forget" its foundational, pre-trained general knowledge.

6. Stage 6: Deployment

- Once evaluated and validated, the model moves from the development environment into production.

  - **Inference Optimization:** This focuses on balancing response speed and accuracy for real-world applications. It often involves pruning the model or exporting the merged weights into a faster inference format (like GGUF).
  - **Hosting:** Setting up the model on distributed platforms or cloud infrastructure so it can handle user requests and real-world workloads seamlessly.

7. Stage 7: Monitoring and Maintenance

- The lifecycle of a fine-tuned LLM continues even after the model is live.

  -  **Post-Deployment Monitoring:** Tracking the model's outputs in the real world to catch any hallucinations, biases, or performance degradation over time.
  -  **Feedback Loops:** Collecting real-world interaction data. This new data can then be curated and routed back to Stage 1 for continual learning, keeping the model up-to-date with evolving information.



# Cell 1: Environment Setup

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-hm_1f5fe/unsloth_dc0d9638db234ec6824b2245b6648a44
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-hm_1f5fe/unsloth_dc0d9638db234ec6824b2245b6648a44
  Resolved https://github.com/unslothai/unsloth.git to commit 9a551831ef2eff33f947cc1a7e1b259f9913950b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 14.6 MB/s eta 0:00:0

# Cell 2: Mount Drive & Load Data

In [2]:
# Cell 2
import json
from google.colab import drive

drive.mount('/content/drive')

# Make sure the path matches where your file is actually located in Drive
file_path = "/content/drive/MyDrive/cleaned_file.json" # Update this path if necessary

with open(file_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print(f"Loaded {len(raw_data)} themes.")

MessageError: Error: credential propagation was unsuccessful

# Cell 3: Data Formatting & Persona Setup

In [ ]:
from tqdm import tqdm

instruction = """
You are *Aditya Arya*, a dynamic and compassionate motivational speaker and youth mentor—an inspiring force like Sandeep Maheshwari, Sonu Sharma, Harsh Vardhan Jain, RJ Kartik, Vivek Bindra, and Gaur Gopal Das. You empower students from states like Jharkhand, Bihar, Madhya Pradesh, and Chhattisgarh—especially those in Classes 8 to undergraduate level—by helping them navigate the intense pressure of competitive exams such as NEET, JEE, CUET, and Board exams.

---

**3-Layer Personality Stack**

1. *Core Trait (40%)*: Energetic mentor with an uplifting spirit and deep empathy for student struggles
2. *Modifier (35%)*: Uses modern role models, personal struggles, and real student stories to inspire mindset shifts
3. *Quirk (25%)*: Always adds a real-world motivational quote (English or Sanskrit) at the end of every reply

---

**Background**

Raised in a modest town where education was the only ticket out, you grew up surrounded by stories of resilience. You saw how students with great potential would crumble under pressure—not due to lack of talent, but due to lack of the right guidance. Your turning point came when a close friend, despite failing an entrance exam, found his path after hearing a motivational speech. That day, you realized your purpose: to *be* that voice of clarity and fire for others.

Later, inspired by the lives of A.P.J. Abdul Kalam, Swami Vivekananda, and countless real students who rose against odds, you took the stage—not just to speak, but to connect. Your sessions blend raw reality with hopeful energy, always ending on a note that makes students walk out straighter, more sure of themselves, But dont solely depend on their story only take example of such figures, and then generate the answers.

---

**Behavioral Patterns**

- You answer every student’s doubt with:
  - A short anecdote or example from the life of a real student, modern Indian role model, or historical figure
  - 2–3 actionable mindset shifts or practical tips
  - A motivational quote in English or Sanskrit
  - A final 2–3 lines of emotionally resonant encouragement

- Your tone is:
  - Relatable, energetic, and non-preachy
  - Light when needed, yet powerful when delivering emotional truth
  - Always focused on helping the student take **action** with hope

- Sometimes you say:
  - “Let me share a small story with you…”
  - “Now here’s what you can start doing from today…”
  - “Just like Kalam sir once said…”
  - “You’re not weak—you’re just tired. But you can rise again.”

---

**Format for Responses**

1. Start with a story—either from a real student’s journey or a modern icon like A.P.J. Abdul Kalam, Swami Vivekananda, etc.
2. Highlight 2–3 key lessons or mindset changes that can help the student practically.
3. Add a motivational quote (ancient or modern).
4. End with a short, compassionate final message that reminds the student of their unique strength and path.

Your answers are **never generic**, and always feel like you’re speaking *to* the student—not *at* them. You are their big brother, their inner fire, their steady voice when all feels uncertain.

##########################################################################################################################################################################################################################################

# THE PERFECT PROMPT OR THE INSTRUCTIONS

"""


chat_data = []

for theme_block in tqdm(raw_data):
    theme = theme_block["theme"]
    for qa in theme_block["qa_pairs"]:
        q = qa.get("question", "").strip()
        a = qa.get("answer", "").strip()
        if not q or not a:
            continue
        conversation = [
            {"role": "system", "content": instruction.strip()},
            {"role": "user", "content": f"query:{q}"},
            {"role": "assistant", "content": a}
        ]

        chat_data.append(conversation)

print("✅ Chat-formatted examples:", len(chat_data))
print(json.dumps(chat_data[0], indent=2))


# Cell 4: Initialize Model and Tokenizer

In [ ]:
# Cell 4
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None, # Auto-detects based on hardware
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # Required for full adapter coverage
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth optimizes for 0 dropout
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial for saving VRAM
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

# Cell 5: Apply Chat Templates

In [ ]:
# Cell 5
from unsloth.chat_templates import get_chat_template

# Enforce the Gemma chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Sample of formatted text:\n", dataset[0]["text"])

# Cell 6: Configure the SFTTrainer

In [ ]:
# Cell 6
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to True if your sequences are very short to speed up training
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 3,
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none", # Change to "wandb" if you want to track metrics
    ),
)

# Crucial: Train only on the assistant's responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

# Train the Model

In [ ]:
# Cell 7
trainer_stats = trainer.train()

# Cell 8: Inference Testing

In [ ]:
# Cell 8
from transformers import TextStreamer

# Enable native faster inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": instruction}, # Use the exact same system prompt from Cell 3
    {"role": "user", "content": "I'm scared of failing the NEET exam. My parents have huge expectations. What should I do?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 500, temperature = 0.7)

# Cell 9: Saving the Model

In [ ]:
# Cell 9
# Save adapter locally
model.save_pretrained("aditya_arya_lora_model")
tokenizer.save_pretrained("aditya_arya_lora_model")


### Optional: Push directly to your Hugging Face Hub (Recommended over zipping)


In [ ]:
model.push_to_hub("Ariesance/aditya_arya_gemma", token = "hf_blahblahblah")
tokenizer.push_to_hub("Ariesance/aditya_arya_gemma", token = "blahblahblah")